------------------------**_Тюнинг самых сильных моделей и составление ансамбля_**--------------------------------------

In [3]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

import pandas as pd
import numpy as np
import optuna
import config
from preprocessing import preprocess_data

from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

In [4]:
# Уровень предупреждений только на Warning
optuna.logging.set_verbosity(optuna.logging.ERROR)

# Загрузка данных и предобработка
df_train = pd.read_csv(config.TRAIN_PATH)
df_test = pd.read_csv(config.TEST_PATH)
test_ids = df_test[config.ID_COL]

df_train_proc, df_test_proc = preprocess_data(df_train, df_test)

X_train = df_train_proc.drop(columns = [config.ID_COL, config.TARGET_COL])
y_train = np.log1p(df_train[config.TARGET_COL])
X_test = df_test_proc.drop(columns = [config.ID_COL])

# Настройка k-fold кросс-валидации при 5 фолдах
kf = KFold(n_splits = config.N_SPLITS, shuffle = config.SHUFFLE, random_state = config.RANDOM_STATE)


print(f"Данные готовы к тюнингу! Признаков: {X_train.shape[1]}, строк: {X_train.shape[0]}")

Данные готовы к тюнингу! Признаков: 208, строк: 1460


In [8]:
import importlib
importlib.reload(config)

# =============================================================
# ТЮНИНГ RANDOM FOREST
# =============================================================
rf_base = RandomForestRegressor(random_state=config.RANDOM_STATE, n_jobs=config.N_JOBS)

# Поиск лучших параметров из конфига
# Запуск поиска: все параметры автоматически подтягиваются из конфига через **
rf_random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=config.RF_PARAM_DIST,
    cv=kf,
    **config.RF_SEARCH_CONFIG  # <- Распакуем n_iter, scoring, verbose, n_jobs, random_state
)

print("Запуск тюнинга Random Forest...")
rf_random_search.fit(X_train, y_train)

# Сохраняем и выводим результаты
best_rf = rf_random_search.best_estimator_
best_rf_rmsle = -rf_random_search.best_score_
print("=" * 50)
print(f"Лучший RMSLE для Random Forest: {best_rf_rmsle:.4f}")
print("Лучшие гиперпараметры:")
for param, val in rf_random_search.best_params_.items():
    print(f"  {param}: {val}")

Запуск тюнинга Random Forest...
Fitting 5 folds for each of 25 candidates, totalling 125 fits
Лучший RMSLE для Random Forest: 0.1386
Лучшие гиперпараметры:
  n_estimators: 300
  min_samples_split: 10
  min_samples_leaf: 1
  max_features: 0.3
  max_depth: 25


In [7]:
importlib.reload(config)
# =============================================================
# ТЮНИНГ LIGHTGBM ЧЕРЕЗ OPTUNA
# =============================================================

def lgbm_objective(trial):
    """Целевая функция: берет диапазоны из config и возвращает средний RMSLE"""

    #  Считываем границы параметров напрямую из config.LGBM_PARAM_DIST:
    num_leaves_min, num_leaves_max = config.LGBM_PARAM_DIST['num_leaves']
    lr_min, lr_max = config.LGBM_PARAM_DIST['learning_rate']
    n_est_min, n_est_max, n_est_step = config.LGBM_PARAM_DIST['n_estimators']
    col_min, col_max = config.LGBM_PARAM_DIST['colsample_bytree']
    sub_min, sub_max = config.LGBM_PARAM_DIST['subsample']
    alpha_min, alpha_max = config.LGBM_PARAM_DIST['reg_alpha']
    lambda_min, lambda_max = config.LGBM_PARAM_DIST['reg_lambda']

     # Optuna выбирает значения строго в диапазонах конфига:
    params = {
        'num_leaves': trial.suggest_int('num_leaves', num_leaves_min, num_leaves_max),
        'learning_rate': trial.suggest_float('learning_rate', lr_min, lr_max, log=True),
        'n_estimators': trial.suggest_int('n_estimators', n_est_min, n_est_max, step=n_est_step),
        'colsample_bytree': trial.suggest_float('colsample_bytree', col_min, col_max),
        'subsample': trial.suggest_float('subsample', sub_min, sub_max),
        'reg_alpha': trial.suggest_float('reg_alpha', alpha_min, alpha_max, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', lambda_min, lambda_max, log=True),
        'random_state': config.RANDOM_STATE,
        'verbosity': -1,
        'n_jobs': config.N_JOBS
    }
    # Оценка модели на 5 фолдах кросс-валидации:
    model = LGBMRegressor(**params)
    fold_scores = []

    for train_idx, val_idx in kf.split(X_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        fold_scores.append(root_mean_squared_error(y_val, preds))

    return np.mean(fold_scores)

# Запуск поиска Optuna
lgbm_study = optuna.create_study(direction="minimize")
print(f"Запуск умного поиска Optuna для LightGBM ({config.OPTUNA_N_TRIALS} попыток)...")
lgbm_study.optimize(lgbm_objective, n_trials=config.OPTUNA_N_TRIALS)
# Итоги и сохранение лучшей модели
best_lgbm_rmsle = lgbm_study.best_value
best_lgbm_params = lgbm_study.best_params
print("=" * 50)
print(f"Лучший RMSLE для LightGBM: {best_lgbm_rmsle:.4f}")
print("Лучшие гиперпараметры:")
for param, val in best_lgbm_params.items():
    print(f"  {param}: {val}")
# Создаем обученный экземпляр лучшей модели
best_lgbm = LGBMRegressor(
    **best_lgbm_params,
    random_state=config.RANDOM_STATE,
    verbosity=-1,
    n_jobs=config.N_JOBS
)

Запуск умного поиска Optuna для LightGBM (30 попыток)...
Лучший RMSLE для LightGBM: 0.1278
Лучшие гиперпараметры:
  num_leaves: 16
  learning_rate: 0.017092276390312633
  n_estimators: 800
  colsample_bytree: 0.4700039616121324
  subsample: 0.6810687557920364
  reg_alpha: 0.002922252692104586
  reg_lambda: 0.011226870431490971


In [10]:
importlib.reload(config)
# =============================================================
# ТЮНИНГ CATBOOST ЧЕРЕЗ OPTUNA
# =============================================================
def catboost_objective(trial):

    # Считываем границы из config.CATBOOST_PARAM_DIST:
    depth_min, depth_max = config.CATBOOST_PARAM_DIST['depth']
    lr_min, lr_max = config.CATBOOST_PARAM_DIST['learning_rate']
    it_min, it_max, it_step = config.CATBOOST_PARAM_DIST['iterations']
    l2_min, l2_max = config.CATBOOST_PARAM_DIST['l2_leaf_reg']
    rs_min, rs_max = config.CATBOOST_PARAM_DIST['random_strength']
    bag_min, bag_max = config.CATBOOST_PARAM_DIST['bagging_temperature']

    # Формируем словарь параметров для текущей попытки:
    params = {
        'depth': trial.suggest_int('depth', depth_min, depth_max),
        'learning_rate': trial.suggest_float('learning_rate', lr_min, lr_max, log=True),
        'iterations': trial.suggest_int('iterations', it_min, it_max, step=it_step),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', l2_min, l2_max, log=True),
        'random_strength': trial.suggest_float('random_strength', rs_min, rs_max, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', bag_min, bag_max),
        'random_state': config.RANDOM_STATE,
        'verbose': 0
    }
    # Оценка на 5 фолдах кросс-валидации:
    model = CatBoostRegressor(**params)
    fold_scores = []

    for train_idx, val_idx in kf.split(X_train):
        X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        fold_scores.append(root_mean_squared_error(y_val, preds))

    return np.mean(fold_scores)


# Запуск оптимизации Optuna
catboost_study = optuna.create_study(direction="minimize")
print(f"Запуск умного поиска Optuna для CatBoost ({config.OPTUNA_N_TRIALS} попыток)...")
catboost_study.optimize(catboost_objective, n_trials=config.OPTUNA_N_TRIALS)


# Итоги и сохранение лучшей модели
best_cat_rmsle = catboost_study.best_value
best_cat_params = catboost_study.best_params
print("=" * 50)
print(f"Лучший RMSLE для CatBoost: {best_cat_rmsle:.4f}")
print("Лучшие гиперпараметры:")
for param, val in best_cat_params.items():
    print(f"  {param}: {val}")


# Создаем лучший экземпляр CatBoost
best_cat = CatBoostRegressor(
    **best_cat_params,
    random_state=config.RANDOM_STATE,
    verbose=0
)

Запуск умного поиска Optuna для CatBoost (30 попыток)...
Лучший RMSLE для CatBoost: 0.1243
Лучшие гиперпараметры:
  depth: 6
  learning_rate: 0.04890977076889204
  iterations: 900
  l2_leaf_reg: 1.3477820796306443
  random_strength: 3.235922782308674
  bagging_temperature: 0.3139841759577428


In [12]:
importlib.reload(config)
# =============================================================
# СБОРКА И ОЦЕНКА АНСАМБЛЯ (VotingRegressor)
# =============================================================
# Создаем ансамбль из наших 3 лучших настроенных моделей
voting_model = VotingRegressor(
    estimators=[
        ('cat', best_cat),
        ('lgbm', best_lgbm),
        ('rf', best_rf)
    ],
    weights=config.VOTING_WEIGHTS,
    n_jobs=config.N_JOBS
)

print("Оцениваем итоговый ансамбль на 5-Fold кросс-валидации...")


# Оцениваем ансамбль через K-Fold (деревьям скейлер не нужен)
voting_fold_scores = []
oof_voting = np.zeros(len(X_train))
for train_idx, val_idx in kf.split(X_train):
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]

    voting_model.fit(X_tr, y_tr)
    preds = voting_model.predict(X_val)

    oof_voting[val_idx] = preds
    voting_fold_scores.append(root_mean_squared_error(y_val, preds))
voting_rmsle = np.mean(voting_fold_scores)
voting_std = np.std(voting_fold_scores)
real_dollars = np.expm1(y_train)
pred_dollars = np.expm1(oof_voting)
voting_mae = mean_absolute_error(real_dollars, pred_dollars)
voting_r2 = r2_score(y_train, oof_voting)


# Формируем финальную сводную таблицу тюнинга:
tuning_summary = [
    {
        "Модель": "Voting Ensemble (Cat + LGBM + RF)",
        "RMSLE": round(voting_rmsle, 4),
        "RMSLE (std)": round(voting_std, 4),
        "MAE ($)": f"${voting_mae:,.2f}",
        "R^2": round(voting_r2, 4)
    },
    {
        "Модель": "CatBoost (Tuned with Optuna)",
        "RMSLE": round(best_cat_rmsle, 4),
        "RMSLE (std)": "-",
        "MAE ($)": "-",
        "R^2": "-"
    },
    {
        "Модель": "LightGBM (Tuned with Optuna)",
        "RMSLE": round(best_lgbm_rmsle, 4),
        "RMSLE (std)": "-",
        "MAE ($)": "-",
        "R^2": "-"
    },
    {
        "Модель": "Random Forest (Tuned with RandomSearch)",
        "RMSLE": round(best_rf_rmsle, 4),
        "RMSLE (std)": "-",
        "MAE ($)": "-",
        "R^2": "-"
    }
]
df_tuning = pd.DataFrame(tuning_summary).sort_values(by="RMSLE").reset_index(drop=True)
df_tuning

Оцениваем итоговый ансамбль на 5-Fold кросс-валидации...


,Модель,RMSLE,RMSLE (std),MAE ($),R^2
0,Voting Ensemble (Cat + LGBM + RF),0.1238,0.0165,"$14,670.28",0.9021
1,CatBoost (Tuned with Optuna),0.1243,-,-,-
2,LightGBM (Tuned with Optuna),0.1278,-,-,-
3,Random Forest (Tuned with RandomSearch),0.1386,-,-,-
